# 🤖 AI Code Reviewer — NLP Pipeline Server
### Google Colab + Flask + ngrok → Spring Boot Integration

This notebook runs a Flask REST server that performs NLP analysis on source code:
- **KeyBERT** — keyphrase extraction
- **transformers** — sentiment analysis (cardiffnlp/twitter-roberta)
- **facebook/bart-large-mnli** — zero-shot intent + topic classification
- **Custom rules** — code smell detection
- **pyngrok** — exposes server to the internet for Spring Boot integration

**Spring Boot reads the ngrok URL from `COLAB_NLP_URL` env variable.**

## Cell 1 — Install dependencies

In [ ]:
# Cell 1: Install all required packages
!pip install -q flask flask-cors pyngrok keybert transformers torch \
             sentence-transformers radon lizard anthropic
print('✅ All packages installed')

## Cell 2 — Import & load models (GPU-accelerated)

In [ ]:
# Cell 2: Imports and model loading
import os, re, json, hashlib, textwrap
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok, conf
from keybert import KeyBERT
from transformers import pipeline
import torch
import radon.complexity as radon_cc
import lizard

device = 0 if torch.cuda.is_available() else -1
print(f'Device: {"GPU ✅" if device == 0 else "CPU ⚠️"}')

print('Loading KeyBERT...')
kw_model = KeyBERT('all-MiniLM-L6-v2')

print('Loading sentiment model...')
sentiment_pipe = pipeline(
    'sentiment-analysis',
    model='cardiffnlp/twitter-roberta-base-sentiment-latest',
    device=device
)

print('Loading zero-shot classifier...')
zero_shot_pipe = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=device
)

print('✅ All models loaded')

## Cell 3 — NLP analysis functions

In [ ]:
# Cell 3: NLP analysis functions

INTENT_LABELS = [
    'Data retrieval service', 'Authentication and security',
    'Data transformation', 'API endpoint handler',
    'Database query', 'Business logic processing',
    'Event handling', 'Configuration management',
    'File I/O operation', 'Utility helper'
]

TOPIC_LABELS = [
    'Spring Boot', 'JPA / Hibernate', 'REST API',
    'Security / Auth', 'Database', 'Testing',
    'Error handling', 'Performance', 'Concurrency',
    'Design patterns', 'Validation', 'Logging'
]

# ── Code smell rules ──────────────────────────────────────────────────────────

SMELL_PATTERNS = {
    'Long method (>50 lines)':      lambda code: code.count('\n') > 50,
    'Magic numbers':                lambda code: bool(re.search(r'(?<![\w\.])\d{2,}(?![\w\.])', code)),
    'Raw SQL string':               lambda code: 'SELECT ' in code.upper() and '"' in code,
    'Empty catch block':            lambda code: bool(re.search(r'catch\s*\([^)]+\)\s*\{\s*\}', code)),
    'System.out.println (use log)': lambda code: 'System.out.println' in code,
    'TODO / FIXME comments':        lambda code: bool(re.search(r'//\s*(TODO|FIXME|HACK|XXX)', code, re.I)),
    'Overly nested code (>4 lvl)':  lambda code: max((len(l) - len(l.lstrip())) // 4
                                                     for l in code.split('\n') if l.strip()) > 4
                                                 if code.strip() else False,
    'Hardcoded credentials':        lambda code: bool(re.search(
        r'(?i)(password|secret|apikey|api_key)\s*=\s*["\'][^"\']{4,}["\']', code)),
    'God class (>20 methods)':      lambda code: code.count('public ') + code.count('private ') > 20,
    'Star import':                  lambda code: bool(re.search(r'import\s+[\w\.]+\*', code)),
}

PATTERN_PATTERNS = {
    'Singleton':      r'getInstance\s*\(',
    'Factory':        r'(?:create|build|make)\w+\s*\(',
    'Builder':        r'\.builder\s*\()',
    'Observer':       r'(?:addListener|addEventListener|subscribe)\s*\(',
    'Repository':     r'@Repository|extends\s+\w*Repository',
    'Service layer':  r'@Service',
    'REST Controller':r'@RestController|@Controller',
    'Dependency Inj.':r'@Autowired|@Inject|constructor\(',
    'Strategy':       r'interface\s+\w*Strategy|implements\s+\w*Strategy',
}


def extract_keyphrases(code: str) -> list[str]:
    """KeyBERT keyphrase extraction — treats code as natural language."""
    # Normalise: strip symbols, keep identifiers
    text = re.sub(r'[{}();,@\[\]]', ' ', code)
    text = re.sub(r'\s+', ' ', text).strip()[:2000]  # cap at 2000 chars
    if len(text) < 20:
        return []
    keywords = kw_model.extract_keywords(
        text, keyphrase_ngram_range=(1, 2),
        stop_words='english', top_n=10, diversity=0.5
    )
    return [kw for kw, score in keywords if score > 0.2]


def analyze_sentiment(code: str) -> str:
    """Sentiment of code comments and string literals."""
    comments = re.findall(r'//[^\n]+|/\*[\s\S]*?\*/|#[^\n]+', code)
    if not comments:
        return 'Neutral'
    sample = ' '.join(comments)[:512]
    result = sentiment_pipe(sample)[0]
    label = result['label'].lower()
    if 'positive' in label:
        return 'Positive'
    if 'negative' in label:
        return 'Negative'
    return 'Neutral'


def classify_intent(code: str) -> str:
    text = re.sub(r'[{}();,@\[\]]', ' ', code)[:800]
    result = zero_shot_pipe(text, INTENT_LABELS, multi_label=False)
    return result['labels'][0]


def classify_topics(code: str) -> list[str]:
    text = re.sub(r'[{}();,@\[\]]', ' ', code)[:800]
    result = zero_shot_pipe(text, TOPIC_LABELS, multi_label=True)
    return [lbl for lbl, score in zip(result['labels'], result['scores']) if score > 0.4][:5]


def detect_smells(code: str) -> list[str]:
    return [name for name, check in SMELL_PATTERNS.items() if check(code)]


def detect_patterns(code: str) -> list[str]:
    found = []
    for name, pattern in PATTERN_PATTERNS.items():
        if re.search(pattern, code):
            found.append(name)
    return found


def compute_complexity(code: str, language: str) -> dict:
    """Uses lizard for cross-language cyclomatic complexity."""
    try:
        ext_map = {'java': '.java', 'python': '.py', 'typescript': '.ts',
                   'javascript': '.js', 'kotlin': '.kt', 'go': '.go', 'sql': '.sql'}
        ext = ext_map.get(language.lower(), '.java')
        analysis = lizard.analyze_file.analyze_source_code('code' + ext, code)
        loc = len([l for l in code.split('\n') if l.strip()])
        avg_cc = sum(f.cyclomatic_complexity for f in analysis.function_list) / max(1, len(analysis.function_list))
        avg_cc = round(avg_cc)
        if avg_cc <= 5:   label = 'Low'
        elif avg_cc <= 10: label = 'Medium'
        elif avg_cc <= 20: label = 'High'
        else:              label = 'Very High'
        # comment density
        comment_lines = len(re.findall(r'^\s*(//|#|\*)', code, re.MULTILINE))
        density = round(comment_lines / max(1, loc) * 100)
        # maintainability index
        import math
        hv = math.log(loc * 4 + 1)
        mi = max(0, min(100, round((171 - 5.2*hv - 0.23*avg_cc - 16.2*math.log(loc+1)) / 171 * 100, 1)))
        return {'cyclomaticComplexity': avg_cc, 'linesOfCode': loc,
                'complexity': label, 'commentDensityPercent': density,
                'maintainabilityIndex': mi}
    except Exception as e:
        loc = len([l for l in code.split('\n') if l.strip()])
        return {'cyclomaticComplexity': 1, 'linesOfCode': loc,
                'complexity': 'Unknown', 'commentDensityPercent': 0,
                'maintainabilityIndex': 50.0}


print('✅ NLP functions defined')

## Cell 4 — Flask REST server

In [ ]:
# Cell 4: Flask server definition

app = Flask(__name__)
CORS(app)


@app.route('/analyze', methods=['POST'])
def analyze():
    """Main NLP analysis endpoint called by Spring Boot."""
    try:
        data = request.get_json(force=True)
        code = data.get('code', '').strip()
        language = data.get('language', 'java').strip().lower()

        if not code:
            return jsonify({'error': 'code is required'}), 400

        # Run all pipeline stages
        metrics   = compute_complexity(code, language)
        keyphrases  = extract_keyphrases(code)
        sentiment   = analyze_sentiment(code)
        intent      = classify_intent(code)
        topics      = classify_topics(code)
        code_smells = detect_smells(code)
        patterns    = detect_patterns(code)

        return jsonify({
            **metrics,
            'sentiment':      sentiment,
            'intent':         intent,
            'keyphrases':     keyphrases,
            'topics':         topics,
            'codeSmells':     code_smells,
            'designPatterns': patterns,
        })

    except Exception as e:
        import traceback
        return jsonify({'error': str(e), 'trace': traceback.format_exc()}), 500


@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'UP', 'service': 'NLP Pipeline', 'gpu': torch.cuda.is_available()})


@app.route('/batch', methods=['POST'])
def batch_analyze():
    """Analyze multiple snippets in one call."""
    data = request.get_json(force=True)
    items = data.get('items', [])
    results = []
    for item in items[:10]:   # max 10 per batch
        try:
            code = item.get('code', '')
            lang = item.get('language', 'java')
            metrics    = compute_complexity(code, lang)
            smells     = detect_smells(code)
            patterns   = detect_patterns(code)
            keyphrases = extract_keyphrases(code)
            results.append({**metrics, 'codeSmells': smells,
                            'designPatterns': patterns, 'keyphrases': keyphrases})
        except Exception as e:
            results.append({'error': str(e)})
    return jsonify({'results': results})


print('✅ Flask app defined — ready to start')

## Cell 5 — Start server + ngrok tunnel

In [ ]:
# Cell 5: Start Flask + expose via ngrok
# ⚠️  Set your ngrok auth token below (free at https://dashboard.ngrok.com)

import threading

NGROK_AUTH_TOKEN = ''  # ← PASTE YOUR NGROK TOKEN HERE

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Kill any existing tunnels
ngrok.kill()

# Start Flask in background thread
def run_flask():
    app.run(port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

# Open ngrok tunnel
import time; time.sleep(2)
public_url = ngrok.connect(5000).public_url

print('='*60)
print('🚀 NLP Pipeline Server is LIVE')
print(f'   Public URL : {public_url}')
print(f'   Health     : {public_url}/health')
print(f'   Analyze    : POST {public_url}/analyze')
print('='*60)
print()
print('📋 Copy this into your Spring Boot application.yml:')
print(f'   nlp.pipeline.colab.base-url: {public_url}')
print()
print('📋 Or set env variable:')
print(f'   export COLAB_NLP_URL={public_url}')

## Cell 6 — Test the server locally

In [ ]:
# Cell 6: Test the NLP pipeline with a sample Spring Boot snippet

import requests

TEST_CODE = """
public class UserService {
    @Autowired
    private UserRepository repo;

    public User findUser(String id) {
        // TODO: add caching
        return repo.findById(id).get();  // no error handling
    }

    public void saveUser(User u) {
        String password = "admin123";  // hardcoded!
        System.out.println("Saving: " + u.getName());
        repo.save(u);
    }
}
"""

resp = requests.post(f'{public_url}/analyze',
                     json={'code': TEST_CODE, 'language': 'java'})

result = resp.json()
print(json.dumps(result, indent=2))

print('\n✅ Test complete')
print(f"   Complexity : {result.get('complexity')}")
print(f"   Smells     : {result.get('codeSmells')}")
print(f"   Patterns   : {result.get('designPatterns')}")
print(f"   Keyphrases : {result.get('keyphrases')}")

## ⚡ Architecture Overview

```
┌──────────────────────────────────────────────────────────┐
│                    SPRING BOOT (Port 8080)                │
│                                                           │
│  POST /api/v1/code-review                                 │
│         │                                                 │
│         ├──[CompletableFuture]──▶ ClaudeLlmService        │
│         │                            └──▶ Anthropic API  │
│         │                                                 │
│         └──[CompletableFuture]──▶ NlpPipelineService      │
│                                      └──▶ ngrok tunnel   │
│                                              │            │
└──────────────────────────────────────────────┼────────────┘
                                               │
                                   ┌───────────▼───────────┐
                                   │  GOOGLE COLAB (Flask)  │
                                   │                        │
                                   │  POST /analyze         │
                                   │  ├─ KeyBERT            │
                                   │  ├─ Sentiment (RoBERTa)│
                                   │  ├─ Intent (BART MNLI) │
                                   │  ├─ Topic classifier   │
                                   │  ├─ Code smell rules   │
                                   │  └─ Design pattern det │
                                   └────────────────────────┘
```